# 🏠 Vision Agent — Solution

This notebook provides a complete working solution to the Vision Agent challenge.
The agent acts as a Casablanca real estate property analyst.

In [ ]:
# ! pip install -r ../../../../requirements.txt -U

In [ ]:
from agent_framework import Message, Content
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity.aio import AzureCliCredential

In [ ]:
import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv("../../../../.env")

In [ ]:
AgentName = "CasaImmo-Vision-Agent"

# SOLUTION — Task 1: Agent Instructions
AgentInstructions = """You are a professional real estate property analyst for Casa Immo, 
a real estate agency based in Casablanca, Morocco.

When given an image of a property interior, you will:
1. Identify and list every piece of furniture and significant decor item you can see,
   with a brief description of each.
2. Assess the overall interior style (e.g. modern, traditional Moroccan, minimalist)
   and quality level (budget, mid-range, premium).
3. Based on the furnishings and quality observed, suggest a realistic monthly rental
   price range in Moroccan Dirhams (MAD) for the Casablanca rental market.

Be concise, factual, and structured in your response."""

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIProjectAgentProvider(
        project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
        credential=credential
    ) as client,
):
    agent = await client.create_agent(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=AgentInstructions,
        name=AgentName
    )

    image_path = "../../files/home.png"

    # SOLUTION — Task 2: Load and encode the image
    with open(image_path, "rb") as image_file:
        image_b64 = base64.b64encode(image_file.read()).decode()
    image_uri = f"data:image/png;base64,{image_b64}"

    # SOLUTION — Task 3: Build a multimodal Message
    message = Message(
        role="user",
        contents=[
            Content.from_text(
                text="Please analyse this property interior. List all furniture items, "
                     "assess the style and quality, and suggest a monthly rental price "
                     "range in MAD for the Casablanca market."
            ),
            Content.from_uri(uri=image_uri, media_type="image/png"),
        ]
    )

    # SOLUTION — Task 4: Run the agent and print the result
    result = await agent.run(message)
    print(f"Agent: {result.text}")